# CycleGAN 训练复现冒烟（ShiftLite / issue #8）

硬边界：自训权重**不进** A/B/C 三组对照；训练结论单独成表。

步骤：① 装环境 → ② 下数据 → ③ 20 iter 冒烟 → ④ 决定跑量 → ⑤ FID/KID 曲线。
免费 T4（16GB）预期：256×256、batch=1、单 epoch≈数分钟量级。

In [ ]:
# ① 环境（Colab 已有 torch；如本地缺包取消注释）
# !pip -q install torch torchvision pillow numpy
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ② 数据：horse2zebra（126MB）
import os, zipfile, urllib.request
URL = 'https://people.eecs.berkeley.edu/~taesung_park/CycleGAN/datasets/horse2zebra.zip'
if not os.path.exists('horse2zebra'):
    urllib.request.urlretrieve(URL, 'horse2zebra.zip')
    zipfile.ZipFile('horse2zebra.zip').extractall('.')
for d in ('trainA', 'trainB', 'testA', 'testB'):
    print(d, len(os.listdir(f'horse2zebra/{d}')))

In [ ]:
# ③ 冒烟：20 iter 验证损失链与 train_losses.csv 落盘（CPU 也能过）
!python train/train.py --data-root horse2zebra --out runs/smoke \
    --n-epochs 1 --max-iters 20
!head -3 runs/smoke/train_losses.csv

In [ ]:
# ④ 正式训练：先 20 epoch 看曲线形状，再决定跑满 100+100
# !python train/train.py --data-root horse2zebra --out runs/exp1 --n-epochs 20 --n-epochs-decay 0
# 跑满论文口径（T4 上数小时量级）：
# !python train/train.py --data-root horse2zebra --out runs/exp1

In [ ]:
# ⑤ FID/KID 曲线（E3 口径：生成批=testA 输出，真实批=testB）
# !python train/evaluate.py --data-root horse2zebra \
#     --ckpt runs/exp1/G_AB_ep20.pth --curve runs/exp1/fid_kid_curve.csv
# !cat runs/exp1/fid_kid_curve.csv

## 记录（回填 train/README.md 表格）
- 免费 GPU 机型 / 单 epoch 耗时 / 总时长 / 与官方权重 FID·KID 差距（单独成表）
- 曲线观察：G/D 博弈是否稳定、cyc 是否下降、身份损失是否收敛